# ARGOS Bot — Ensemble Training (Colab GPU)

Train LSTM + XGBoost + MetaModel + Calibrator for multiple symbols.

## Workflow
1. Upload `data_export.zip` to Colab (or mount Drive)
2. This notebook trains everything with GPU
3. Download the resulting `checkpoints.zip` and import via `python -m scripts.import_checkpoint --input checkpoints.zip`

**Requires**: TensorFlow, XGBoost, scikit-learn, pandas, numpy, pyarrow

In [ ]:
# Install deps (run once)
!pip install tensorflow xgboost scikit-learn pandas numpy pyarrow -q

In [ ]:
import hashlib
import json
import os
import shutil
import tempfile
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

print('TensorFlow:', tf.__version__)
print('GPU:', 'YES' if tf.config.list_physical_devices('GPU') else 'NO')

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
# ── Setup ──────────────────────────────────────────────────────

# Paths
DATA_ZIP = 'data_export.zip'  # Upload this file
OUTPUT_ZIP = 'checkpoints.zip'
WORK_DIR = Path('/content/argos_training')

# LSTM params (from ModelConfig defaults)
LOOKBACK = 60
BATCH_SIZE = 32
MAX_EPOCHS = 200
DROPOUT = 0.2
EARLY_STOP_PATIENCE = 10
CONFIDENCE_THRESHOLD = 0.7

# Walk Forward
N_WINDOWS = 5
WINDOW_VAL_PCT = 0.15

WORK_DIR.mkdir(parents=True, exist_ok=True)

## 1. Extract data

In [ ]:
with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
    zf.extractall(str(WORK_DIR))

with open(WORK_DIR / 'manifest.json') as f:
    manifest = json.load(f)

print('Manifest:', json.dumps(manifest, indent=2)[:500])

In [ ]:
symbols = manifest['symbols']
print(f'Loaded {len(symbols)} symbols:')
for s in symbols:
    print(f"  {s['symbol']}: {s['n_samples']} samples, {s['n_features']} features")

## 2. Define building blocks

In [ ]:
def build_lstm(n_features: int) -> tf.keras.Model:
    """Build LSTM architecture matching NovaQuantKerasModel."""
    inputs = tf.keras.Input(shape=(LOOKBACK, n_features), name='ohlcv_window')
    x = tf.keras.layers.LSTM(128, return_sequences=True, name='lstm_0')(inputs)
    x = tf.keras.layers.Dropout(DROPOUT, name='dropout_0')(x)
    x = tf.keras.layers.LSTM(64, return_sequences=True, name='lstm_1')(x)
    x = tf.keras.layers.Dropout(DROPOUT, name='dropout_1')(x)
    x = tf.keras.layers.LSTM(32, name='lstm_2')(x)
    x = tf.keras.layers.Dropout(DROPOUT, name='dropout_2')(x)
    x = tf.keras.layers.Dense(16, activation='relu', name='dense')(x)
    x = tf.keras.layers.Dropout(DROPOUT, name='dropout_dense')(x)
    outputs = tf.keras.layers.Dense(3, activation='softmax', name='output')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name='NovaQuant')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


def make_windows(features: np.ndarray) -> np.ndarray:
    """Sliding window view: (n, lookback, n_features)."""
    windows = np.lib.stride_tricks.sliding_window_view(features, LOOKBACK, axis=0)
    return windows.transpose(0, 2, 1)

In [ ]:
def train_lstm(
    x_train: np.ndarray, y_train: np.ndarray,
    x_val: np.ndarray, y_val: np.ndarray,
    n_features: int,
) -> tuple[tf.keras.Model, dict]:
    """Train LSTM with early stopping."""
    model = build_lstm(n_features)

    cb = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=EARLY_STOP_PATIENCE,
            restore_best_weights=True, verbose=0,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=EARLY_STOP_PATIENCE // 2, min_lr=1e-6, verbose=0,
        ),
    ]

    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
        callbacks=cb, verbose=1,
    )

    val_loss = float(min(history.history['val_loss']))
    best_idx = history.history['val_loss'].index(val_loss)
    val_acc = float(history.history['val_accuracy'][best_idx])

    metrics = {
        'val_loss': val_loss,
        'val_accuracy': val_acc,
        'epochs_trained': len(history.history['loss']),
    }
    return model, metrics

In [ ]:
def train_xgboost(
    x_train: np.ndarray, y_train: np.ndarray,
    x_val: np.ndarray, y_val: np.ndarray,
) -> tuple['xgb.XGBClassifier', dict]:
    """Train XGBoost classifier."""
    n = x_train.shape[0] * x_train.shape[1]
    x_train_2d = x_train.reshape(x_train.shape[0], -1)
    x_val_2d = x_val.reshape(x_val.shape[0], -1)
    y_train_lab = np.argmax(y_train, axis=1)
    y_val_lab = np.argmax(y_val, axis=1)

    model = xgb.XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        objective='multi:softprob', num_class=3,
        eval_metric='mlogloss', early_stopping_rounds=10,
        verbosity=0,
    )
    model.fit(
        x_train_2d, y_train_lab,
        eval_set=[(x_val_2d, y_val_lab)],
        verbose=False,
    )

    val_pred = model.predict_proba(x_val_2d)
    val_acc = float(np.mean(np.argmax(val_pred, axis=1) == y_val_lab))

    return model, {'val_accuracy': val_acc}

In [ ]:
def predict_lstm(model: tf.keras.Model, x: np.ndarray) -> tuple[float, float, float]:
    """LSTM forward pass -> (buy, sell, hold) probs."""
    batch = np.expand_dims(x, axis=0)
    probs = model.predict(batch, verbose=0)[0]
    return (float(probs[0]), float(probs[1]), float(probs[2]))


def predict_xgb(model: 'xgb.XGBClassifier', x: np.ndarray) -> tuple[float, float, float]:
    """XGBoost forward pass -> (buy, sell, hold) probs."""
    flat = x.reshape(1, -1)
    probs = model.predict_proba(flat)[0]
    return (float(probs[0]), float(probs[1]), float(probs[2]))


def signal_to_probs(probs: tuple[float, float, float]) -> tuple[float, float, float]:
    """Normalise to sum=1."""
    total = sum(probs)
    if total > 0:
        return tuple(p / total for p in probs)
    return (1/3, 1/3, 1/3)

## 3. Train per symbol

In [ ]:
import io
import joblib

CHECKPOINT_DIR = WORK_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

results = []

for sym_info in symbols:
    key = sym_info['key']
    symbol = sym_info['symbol']
    print(f'\n{"="*60}')
    print(f'Training {symbol} ({key})')
    print(f'{"="*60}')

    # Load raw features (unnormalized)
    feat_df = pd.read_parquet(WORK_DIR / f'features/{key}_features.parquet')
    tgt_df = pd.read_parquet(WORK_DIR / f'features/{key}_targets.parquet')
    features_raw = feat_df.values.astype(np.float32)
    targets = tgt_df.values.astype(np.float32)
    n_features = features_raw.shape[1]

    # Map column names to indices dynamically
    cols = feat_df.columns
    bb_upper_idx = cols.get_loc('bb_upper')
    bb_middle_idx = cols.get_loc('bb_middle')
    bb_lower_idx = cols.get_loc('bb_lower')
    adx_idx = cols.get_loc('adx')
    atr_idx = cols.get_loc('atr')
    rsi_idx = cols.get_loc('rsi')
    volume_idx = cols.get_loc('volume')

    # Windows on raw features
    windows = make_windows(features_raw)
    aligned = targets[LOOKBACK - 1:]
    aligned = aligned[:len(windows)]

    n = len(windows)
    window_size = n // N_WINDOWS

    all_meta_features = []
    all_oof_targets = []
    lstm_models = []
    xgb_models = []

    # ── Walk Forward (per-window normalization) ──────────────
    for w in range(N_WINDOWS):
        print(f'\n  Window {w+1}/{N_WINDOWS}')
        start = w * window_size
        end = n if w == N_WINDOWS - 1 else (w + 1) * window_size

        w_data_raw = windows[start:end]
        w_tgts = aligned[start:end]
        n_total = len(w_data_raw)
        n_val = max(1, int(n_total * WINDOW_VAL_PCT))
        n_train = n_total - n_val

        # Split before normalising
        x_tr_raw = w_data_raw[:n_train]
        x_va_raw = w_data_raw[n_train:]
        y_tr = w_tgts[:n_train]
        y_va = w_tgts[n_train:]

        # Per-window z-score: compute from train ONLY
        w_means = np.mean(x_tr_raw, axis=(0, 1))
        w_stds = np.std(x_tr_raw, axis=(0, 1)) + 1e-6

        x_tr = (x_tr_raw - w_means) / w_stds
        x_va = (x_va_raw - w_means) / w_stds

        print(f'    Train: {len(x_tr)}, Val: {len(x_va)}')

        # LSTM
        lstm, lstm_metrics = train_lstm(x_tr, y_tr, x_va, y_va, n_features)
        print(f'    LSTM val_acc: {lstm_metrics["val_accuracy"]:.4f}')

        # XGBoost
        xgb_model, xgb_metrics = train_xgboost(x_tr, y_tr, x_va, y_va)
        print(f'    XGB val_acc: {xgb_metrics["val_accuracy"]:.4f}')

        # OOF predictions on validation split
        oof_lstm = np.array([
            signal_to_probs(predict_lstm(lstm, x_va[i]))
            for i in range(len(x_va))
        ])
        oof_xgb = np.array([
            signal_to_probs(predict_xgb(xgb_model, x_va[i]))
            for i in range(len(x_va))
        ])

        # Meta features with corrected global_idx + dynamic indices + real BBW
        for i in range(len(x_va)):
            global_idx = start + n_train + i
            raw_row = features_raw[global_idx]
            bbw = float((raw_row[bb_upper_idx] - raw_row[bb_lower_idx]) /
                        (raw_row[bb_middle_idx] + 1e-10))
            adx_val = float(raw_row[adx_idx]) if n_features > adx_idx else 25.0
            atr_val = float(raw_row[atr_idx]) if n_features > atr_idx else 100.0
            rsi_val = float(raw_row[rsi_idx]) if n_features > rsi_idx else 50.0
            vol_val = float(raw_row[volume_idx]) if n_features > volume_idx else 1000.0
            mf = [*oof_lstm[i], *oof_xgb[i], adx_val, bbw, atr_val, rsi_val, vol_val]
            all_meta_features.append(mf)
            all_oof_targets.append(y_va[i])

        lstm_models.append(lstm)
        xgb_models.append(xgb_model)

        # Save last window normalisation stats for inference
        last_w_means = w_means.copy()
        last_w_stds = w_stds.copy()

    # ── MetaModel (XGBoost stacking) ──────────────────────────
    print('\n  Training MetaModel (XGBoost stacking)...')
    meta_features = np.array(all_meta_features)
    meta_targets = np.array(all_oof_targets)
    meta_labels = np.argmax(meta_targets, axis=1)

    meta_model = xgb.XGBClassifier(
        n_estimators=50, max_depth=3, learning_rate=0.1,
        objective='multi:softprob', num_class=3,
        eval_metric='mlogloss', verbosity=0,
    )
    meta_model.fit(meta_features, meta_labels)
    meta_pred = meta_model.predict_proba(meta_features)
    meta_acc = float(np.mean(np.argmax(meta_pred, axis=1) == meta_labels))
    print(f'    MetaModel train_acc: {meta_acc:.4f}')

    # ── Calibrator (Platt Scaling, binary BUY) ────────────────
    print('  Training Calibrator...')
    avg_probs = (all_meta_features[:, :3] + all_meta_features[:, 3:6]) / 2.0
    max_probs = np.max(avg_probs, axis=1)
    calibrator = LogisticRegression(C=1.0, solver='lbfgs')
    calibrator.fit(max_probs.reshape(-1, 1), (meta_labels == 0).astype(int))

    # ── Save final model (last window) ────────────────────────
    final_lstm = lstm_models[-1]
    final_xgb = xgb_models[-1]

    # Serialise weights
    weights_buf = io.BytesIO()
    final_lstm.save(weights_buf, save_format='keras')
    weights_bytes = weights_buf.getvalue()
    weights_hash = hashlib.sha256(weights_bytes).hexdigest()[:16]

    version = f'v1.0.{int(datetime.now(timezone.utc).timestamp())}'
    version_dir = CHECKPOINT_DIR / key / version
    version_dir.mkdir(parents=True, exist_ok=True)

    # model_config.json
    config = {
        'lookback': LOOKBACK,
        'confidence_threshold': CONFIDENCE_THRESHOLD,
        'layers': [128, 64, 32, 16],
        'features': list(feat_df.columns),
        'target_lookahead': 5,
        'target_return_pct': 1.0,
        'dropout_rate': DROPOUT,
        'batch_size': BATCH_SIZE,
        'max_epochs': MAX_EPOCHS,
        'early_stop_patience': EARLY_STOP_PATIENCE,
    }
    with open(version_dir / 'model_config.json', 'w') as f:
        json.dump(config, f, indent=2)

    # model_metadata.json — stores LAST WINDOW'S normalisation stats
    metadata = {
        'model_version': version,
        'trained_at': datetime.now(timezone.utc).isoformat(),
        'weights_hash': weights_hash,
        'feature_means': [float(m) for m in last_w_means],
        'feature_stds': [float(s) for s in last_w_stds],
        'metrics': {
            'lstm_val_accuracy': lstm_metrics['val_accuracy'],
            'xgb_val_accuracy': xgb_metrics['val_accuracy'],
            'meta_train_accuracy': meta_acc,
        },
    }
    with open(version_dir / 'model_metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)

    # weights.keras
    with open(version_dir / 'weights.keras', 'wb') as f:
        f.write(weights_bytes)

    # Extra: xgboost model, meta model, calibrator
    final_xgb.save_model(str(version_dir / 'xgboost_model.json'))
    meta_model.save_model(str(version_dir / 'meta_model.json'))
    joblib.dump(calibrator, version_dir / 'calibrator.pkl')

    results.append({
        'symbol': symbol,
        'key': key,
        'version': version,
        'lstm_acc': lstm_metrics['val_accuracy'],
        'xgb_acc': xgb_metrics['val_accuracy'],
        'meta_acc': meta_acc,
        'n_samples': len(all_oof_targets),
    })

    print(f'\n  ✅ {symbol} → {version}')
    print(f'     LSTM: {lstm_metrics["val_accuracy"]:.4f} | XGB: {xgb_metrics["val_accuracy"]:.4f} | Meta: {meta_acc:.4f}')

print('\n' + '='*60)
print('Training complete!')
print('='*60)

In [ ]:
# Summary table
import pandas as pd
summary = pd.DataFrame(results)
print(summary.to_string(index=False))
print(f'\nSymbols with accuracy > 0.33: {(summary["meta_acc"] > 0.33).sum()}/{len(summary)}')

## 4. Export checkpoints

In [ ]:
print(f'Creating {OUTPUT_ZIP}...')

with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(CHECKPOINT_DIR):
        for fname in files:
            path = Path(root) / fname
            arcname = str(path.relative_to(CHECKPOINT_DIR.parent.parent))
            zf.write(path, arcname)

size_mb = os.path.getsize(OUTPUT_ZIP) / (1024 * 1024)
print(f'Created {OUTPUT_ZIP} ({size_mb:.1f} MB)')
print('\nDownload this file and import with:')
print('  python -m scripts.import_checkpoint --input checkpoints.zip')

In [ ]:
# Download trigger (Colab)
from google.colab import files
files.download(OUTPUT_ZIP)